In [2]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import yaml
import logging as log
import os

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

- Set the LOG

In [3]:
# Check if the directory exists
if not os.path.exists('log'):
    # If it doesn't exist, create it
    os.mkdir('log')
    print("Directory 'log' created successfully.")
else:
    # If it already exists, print a message
    print("Directory 'log' already exists.")
    
import logging as log
log.basicConfig(level=log.INFO, format='%(asctime)s - %(levelname)s - %(message)s' , datefmt='%Y-%m-%d %H:%M:%S')
file_handler = log.FileHandler('log/data_preparation_log.txt')
# Add the FileHandler to the root logger
log.getLogger().addHandler(file_handler)

Directory 'log' already exists.


- Load CONFIG

In [4]:
def load_config(
        file_path:str)-> dict:
    """
    Readsn the user configuration file and return (load) as a Dictionary
    """

    with open(file_path, 'r') as file:
        data = yaml.safe_load(file)

    return data

In [5]:
data_sources_config_file:str='config/data_sources.yml'
data_sources:dict=load_config(data_sources_config_file)
current_region=data_sources['Regions']['region_1']
_CRC_=current_region['code'] # Current Region Code (CRC), later to be configured with the user config file.

In [6]:
CODERS_data:dict= data_sources['CODERS']
url=CODERS_data['source']['url_1']
api_elias=CODERS_data['source']['api_key']['Elias']
query="?key="+api_elias
data_save_to=CODERS_data['data_save_to']

-  Create Directories

In [7]:
# List of directories to create
directories=[
    'datapackage/data_supply_chain/CODERS',
    'vis',
    ]

# Create directories from the list
for directory in directories:
    # directory_created=utility.create_directories(path)
    os.makedirs(os.path.join(directory,_CRC_),exist_ok=True)

## Tables

-  Create Directories

In [8]:
# List of directories to create
directoires_core=['datapackage/data_supply_chain/CODERS',]

# Create directories from the list
for directory in directoires_core:
    # directory_created=utility.create_directories(path)
    os.makedirs(os.path.join(directory),exist_ok=True)

In [9]:
tables_list = [t for t in requests.get(url+"/tables"+query).json()]
tables_list

['coders', 'cef']

# Pull Data from CODERS 

### Generators

In [36]:
table_name="generators"
generators_df:pd.DataFrame=pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query).json())

province_mask=generators_df['province']==_CRC_
province_generators:pd.DataFrame=generators_df[province_mask]

data=province_generators
file_path:str=os.path.join(data_save_to,_CRC_,f'{_CRC_}_{table_name}.pkl')
data.to_pickle(file_path)

### Wind Generators

In [55]:
table_name='wind_generators'
wind_generators = pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query).json())

province_mask=wind_generators['province']==_CRC_
province_wind_generators=wind_generators[province_mask]

data=province_wind_generators
file_path:str=os.path.join(data_save_to,_CRC_,f'{_CRC_}_{table_name}.pkl')
data.to_pickle(file_path)

In [56]:
province_wind_generators

,generation_facility_name,generation_facility_code,owner,province,location,latitude,longitude,elevation,copper_balancing_area,operating_region,...,capacity_adjustment,facility_effective_capacity,capacity_factor,average_annual_energy,generation_units_represented,total_facility_generation_units,wind_turbine_capacity,project_area,density,notes
15,Bear Mountain,BC_BMW_GEN,Bear Mountain Wind Limited Partnership,BC,Dawson Creek,55.704126,-120.426360,933,British Columbia.a,British Columbia,...,0.2400,25,0.2194,197.00,34,34,3.00,18.13,5.65,NULL
34,Cape Scott,BC_CSS_GEN,Cape Scott Wind LP,BC,Port Hardy,50.789021,-128.003295,313,British Columbia.a,British Columbia,...,0.2400,24,0.3640,316.30,55,55,1.80,44.77,2.22,NULL
61,Dokie,BC_DKW_GEN,Dokie General Partnership,BC,Chetwynd,55.811775,-122.247353,1303,British Columbia.a,British Columbia,...,0.2400,35,0.2973,375.00,48,48,3.00,250.00,0.58,NULL
142,Meikle,BC_MKL_GEN,Meikle Wind Energy Limited Partnership,BC,Tumbler Ridge,55.285560,-121.469056,1189,British Columbia.a,British Columbia,...,0.2400,47,0.3169,541.00,61,61,3.20,44.70,4.36,NULL
151,Moose Lake,BC_MLW_GEN,Moose Lake Wind Limited Partnership,BC,Tumbler Ridge,55.285633,-121.298571,1158,British Columbia.a,British Columbia,...,0.2400,4,0.4259,55.96,4,4,3.75,2.77,5.42,GATOR 8015535
167,Pennask,BC_PSW_GEN,Canadian Power Holdings,BC,Westbank,49.925611,-120.109281,1574,British Columbia.a,British Columbia,...,0.2400,4,0.3790,49.80,5,5,3.00,5.10,2.94,NULL
179,Quality,BC_QTY_GEN,Capital Power L.P.,BC,Tumbler Ridge,55.184599,-120.854677,1235,British Columbia.a,British Columbia,...,0.2400,34,0.3829,476.91,79,79,1.80,30.62,4.64,NULL
196,Shinish Creek,BC_SHC_GEN,Zero Emissions Shinish Creek Limited Partnership,BC,Summerland,49.660175,-120.122212,1822,British Columbia.a,British Columbia,...,0.2400,4,0.4148,54.50,5,5,3.00,8.00,1.88,GATOR 3412506
215,Sukunka,BC_SUK_GEN,Sukunka Wind Project Limited Partnership,BC,Chetwynd,55.543022,-121.563656,939,British Columbia.a,British Columbia,...,0.2400,4,0.3805,50.00,4,4,3.75,19.74,0.76,iMap BC
250,Zonnebeke,BC_ZBK_GEN,Zonnebeke Wind Project Limited Partnership,BC,Chetwynd,55.589825,-121.565488,771,British Columbia.a,British Columbia,...,0.2400,4,0.3805,50.00,4,4,3.75,19.74,0.76,iMap BC


### CF

In [11]:
for gen_type_copper, df in province_generators.groupby("gen_type_copper"):
    print(gen_type_copper, df["capacity_factor"].unique())

NG_CC ['0.8235' '0.2271' '0.9548' '0.7991']
NG_SC ['0.2654' '0.2667']
biomass ['0.3208' '0.9549' '0.2119' '0.6976' '0.8464' '0.1460' '0.4083' '0.7058'
 '0.2118' '0.9167' '0.8662' '0.8866' '0.0533' '0.4337' '0.2306' '0.9196'
 '0.8390' '0.2629' '0.4330' '0.6718' '0.1921' '0.6469' '0.2120' '0.2011'
 '0.8990' '0.2366' '0.2123' '0.7641' '0.7801' '0.7096' '0.0839' '0.8302'
 '0.8019']
hydro_daily ['0.3312' '0.4733' '0.5115' '0.3280' '0.3025' '0.9037' '0.4078' '0.4151'
 '0.6120' '0.5333' '0.3334' '0.3357' '0.3380' '0.5619' '0.4281' '0.3751'
 '0.6579' '0.2138' '0.4558' '0.4832']
hydro_monthly ['0.6006' '0.4213' '0.4494' '0.6221' '0.4018' '0.5410']
hydro_run ['0.4623' '0.6147' '0.4693' '0.4165' '0.5122' '0.5274' '0.8368' '0.5413'
 '0.3681' '0.4566' '0.3624' '0.4460' '0.4481' '0.3362' '0.3425' '0.6469'
 '0.5625' '0.4998' '0.5673' '0.4616' '0.5137' '0.4281' '0.5466' '0.4336'
 '0.5202' '0.3900' '0.5946' '0.3840' '0.4005' '0.3206' '0.3984' '0.5100'
 '0.4438' '0.4981' '0.3678' '0.4920' '0.5858' '0.44

### Forecasted Annual Demand

In [37]:
table_name='forecasted_annual_demand'
forecasted_annual_demand:pd.DataFrame=pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query).json())

province_mask=forecasted_annual_demand['province']==current_region['name']
province_forecasted_annual_demand=forecasted_annual_demand[province_mask]
province_forecasted_annual_demand

data=province_forecasted_annual_demand
file_path:str=os.path.join(data_save_to,_CRC_,f'{_CRC_}_{table_name}.pkl')
data.to_pickle(file_path)

### Forecasted Peak Demand

In [38]:
table_name='forecasted_peak_demand'
forecasted_peak_demand:pd.DataFrame=pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query).json())

province_mask=forecasted_peak_demand['province']==current_region['name']
province_forecasted_peak_demand=forecasted_peak_demand[province_mask]
province_forecasted_peak_demand

data=province_forecasted_peak_demand
file_path:str=os.path.join(data_save_to,_CRC_,f'{_CRC_}_{table_name}.pkl')
data.to_pickle(file_path)

In [40]:
province_forecasted_peak_demand

,province,unit,2020_MW,2021_MW,2022_MW,2023_MW,2024_MW,2025_MW,2026_MW,2027_MW,...,2042_MW,2043_MW,2044_MW,2045_MW,2046_MW,2047_MW,2048_MW,2049_MW,2050_MW,notes
1,British Columbia,MW,10403,10682,10881,11026,11124,11098,11180,11235,...,12586,12700,12816,12932,13050,13169,13289,13409,13531,None


### Demand Profile

-  Create Directories

In [58]:
table_name='provincial_demand'
demand_dataset = pd.DataFrame.from_dict(requests.get(url + f"/{table_name}" + query).json())

In [59]:
province_mask=demand_dataset['province']==_CRC_
province_demand_dataset=demand_dataset[province_mask] 
print(F"Demand Profile dataset for province - {current_region['name']} available for the follolwing years -\n{province_demand_dataset.year.values}")
print(f"Please select the year.")

Demand Profile dataset for province - British Columbia available for the follolwing years -
['2018,2019,2020,2021,2022']
Please select the year.


In [63]:
year=2022
province_demand_profile_yr = pd.DataFrame.from_dict(requests.get(url + f"/{table_name}" + query + f"&year=2020&province={_CRC_}").json())

In [64]:
province_demand_profile_yr.set_index('local_time',inplace=True)
province_demand_profile_yr.index = pd.to_datetime(province_demand_profile_yr.index)
province_demand_profile_yr

data=province_demand_profile_yr

file_path:str=os.path.join(data_save_to,_CRC_,f'{_CRC_}_{table_name}_profile.pkl')
data.to_pickle(file_path)

In [23]:
# Resampling the data
df=province_demand_profile_yr
hourly_df = df
daily_df = df.resample('D').sum()
weekly_df = df.resample('W').sum()
monthly_df = df.resample('M').sum()
quarterly_df = df.resample('Q').sum()

# Create a figure
fig = make_subplots(rows=1, cols=1)

# Add traces for each aggregation type
fig.add_trace(go.Scatter(x=hourly_df.index, y=hourly_df['demand_MWh'], mode='lines', name='Hourly'), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_df.index, y=daily_df['demand_MWh'], mode='lines', name='Daily', visible='legendonly'), row=1, col=1)
fig.add_trace(go.Scatter(x=weekly_df.index, y=weekly_df['demand_MWh'], mode='lines', name='Weekly', visible='legendonly'), row=1, col=1)
fig.add_trace(go.Scatter(x=monthly_df.index, y=monthly_df['demand_MWh'], mode='lines', name='Monthly', visible='legendonly'), row=1, col=1)
fig.add_trace(go.Scatter(x=quarterly_df.index, y=quarterly_df['demand_MWh'], mode='lines', name='Quarterly', visible='legendonly'), row=1, col=1)


# Define labels and ticks
daily_ticks = hourly_df.index[::12]    # Every 36 hours
daily_ticks = daily_df.index[::10]    # Every 10 days
weekly_ticks = weekly_df.index[::3]  # Every 3 weeks
monthly_ticks = monthly_df.index[::1]  # Every month

# Add dropdown menu
fig.update_layout(
    updatemenus=[{
        'buttons': [
            {'label': 'Hourly', 'method': 'update', 'args': [
                {'visible': [True, False, False, False, False]},
                {'xaxis': {'title': 'Time', 'tickvals': daily_ticks, 'ticktext': daily_ticks.strftime('%Y-%m-%d %H:%M:%S')}},
                {'yaxis': {'title': 'Demand (MWh)'}}
            ]},
            {'label': 'Daily', 'method': 'update', 'args': [
                {'visible': [False, True, False, False, False]},
                {'xaxis': {'title': 'Date', 'tickvals': daily_ticks, 'ticktext': daily_ticks.strftime('%Y-%m-%d')}},
                {'yaxis': {'title': 'Demand (MWh)'}}
            ]},
            {'label': 'Weekly', 'method': 'update', 'args': [
                {'visible': [False, False, True, False, False]},
                {'xaxis': {'title': 'Week', 'tickvals': weekly_ticks, 'ticktext': weekly_ticks.strftime('%Y-W%U')}},
                {'yaxis': {'title': 'Demand (MWh)'}}
            ]},
            {'label': 'Monthly', 'method': 'update', 'args': [
                {'visible': [False, False, False, True, False]},
                {'xaxis': {'title': 'Month', 'tickvals': monthly_ticks, 'ticktext': monthly_ticks.strftime('%Y-%m')}},
                {'yaxis': {'title': 'Demand (MWh)'}}
            ]},
            {'label': 'Quarterly', 'method': 'update', 'args': [
                {'visible': [False, False, False, False, True]},
                {'xaxis': {'title': 'Quarter', 'tickvals': quarterly_df.index, 'ticktext': quarterly_df.index.strftime('%Y-Q%q')}},
                {'yaxis': {'title': 'Demand (MWh)'}}
            ]}
        ],
        'direction': 'down',
        'showactive': True
    }],
    title='Demand in MWh over Time',
    xaxis_title='Time',
    yaxis_title='Demand (MWh)'
)

# Save the plot to an HTML file
fig.write_html(f'vis/{_CRC_}/Demand_profile_{_CRC_}_{year}.html')

# Display the plot
pio.show(fig)

### Storage

In [67]:
table_name='storage'
storage_df = pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query).json())


province_mask=storage_df['province']==_CRC_
province_storage_df=storage_df[province_mask]
if(len(province_storage_df)==0):
    print(f"No storage found for province - {current_region['name']}")
else:
    data=province_storage_df

    file_path:str=os.path.join(data_save_to,_CRC_,f'{_CRC_}_{table_name}_profile.pkl')
    data.to_pickle(file_path)

No storage found for province - British Columbia


In [68]:
table_name='generation_planning_reserve'
generation_planning_reserve= pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query).json())

# Generation Planning Reserve

In [69]:
table_name='generation_planning_reserve'
generation_planning_reserve = pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query).json())

province_mask=generation_planning_reserve['province']==current_region['name']
province_generation_planning_reserve=generation_planning_reserve[province_mask]

data=province_generation_planning_reserve
file_path:str=os.path.join(data_save_to,_CRC_,f'{_CRC_}_{table_name}.pkl')
data.to_pickle(file_path)

In [70]:
province_generation_planning_reserve

,province,winter_wind,winter_solar,summer_wind,summer_solar
1,British Columbia,0.42,0.02,0.25,0.36


In [75]:
# table_name='annual_demand_and_efficiencies'
# annual_demand_and_efficiencies = pd.DataFrame.from_dict(requests.get(url+f"/{table_name}"+query+"/attributes").json())